# DINOv3 ViT — unsupervised chinee apple localization**Stage 1: model first, labels later.** No manual annotation. We run a frozenDINOv3 ViT-B/16 backbone, cluster its per-patch features with KMeans, and youassign *one integer* — "cluster N is chinee apple" — after looking at theoverlay. Those cluster ids are the pseudo-labels for the linear probe in stage 2.```tile -> DINOv3 patch tokens -> L2-norm -> KMeans(k) -> pick cluster -> mask + heat + boxes```Set **Runtime -> Change runtime type -> GPU (T4)** before running.

## 1. Setup — deps + this repo

In [ ]:
!pip install -q scikit-learn scipy matplotlibimport torchprint('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
import os, sysREPO = '/content/chinee-apple-cv-benchmark'if not os.path.isdir(REPO):    !git clone -q https://github.com/nutboltu/chinee-apple-cv-benchmark.git {REPO}sys.path.insert(0, REPO)                                   # shared backbones packagesys.path.insert(0, os.path.join(REPO, 'pipelines', 'dinov3-kmeans-cluster'))from backbones import dino as vfimport clustering as clprint('backbone from', vf.__file__)print('clustering from', cl.__file__)

## 2. Load the backbone**DINOv3 is gated.** Accept Meta's license, download the ViT-B/16 weights(https://ai.meta.com/resources/models-and-libraries/dinov3-downloads/), then in acell run `!git clone https://github.com/facebookresearch/dinov3.git /content/dinov3`,upload the `.pth`, and set `FAMILY='dinov3'` with the two paths below.**Until then, `FAMILY='dinov2'`** loads an ungated ViT-B/14 from `torch.hub` andruns the identical pipeline — swap later with zero code changes.

In [ ]:
FAMILY = 'dinov2'            # 'dinov3' (gated) or 'dinov2' (ungated, works today)REPO_DINOV3 = None          # e.g. '/content/dinov3'WEIGHTS_DINOV3 = None       # e.g. '/content/dinov3_vitb16_pretrain_lvd1689m.pth'if FAMILY == 'dinov3' and not (REPO_DINOV3 and WEIGHTS_DINOV3):    raise SystemExit("DINOv3 is gated: set REPO_DINOV3 + WEIGHTS_DINOV3, or use FAMILY='dinov2'.")model, meta = vf.load_backbone(FAMILY, repo=REPO_DINOV3, weights=WEIGHTS_DINOV3)print('loaded', meta)

## 3. Upload a UAV tile

In [ ]:
from google.colab import filesup = files.upload()IMAGE = next(iter(up))print('using', IMAGE)

## 4. Extract features + cluster (the annotation-free step)

In [ ]:
import matplotlib.pyplot as pltK = 12feats, grid = vf.patch_features(model, IMAGE, meta, img_size=768)labels, centroids = vf.cluster_patches(feats, k=K)print('patches:', feats.shape, '| grid:', grid)overlay = vf.cluster_overlay(IMAGE, labels, grid, K)plt.figure(figsize=(9, 9)); plt.imshow(overlay); plt.axis('off')plt.title(f'KMeans clusters (k={K})'); plt.show()

### 4b. Per-cluster panels — makes the pick obvious

In [ ]:
import numpy as npfrom PIL import Imagebase = np.asarray(Image.open(IMAGE).convert('RGB'))W, H = Image.open(IMAGE).sizelbl_full = cl.upsample_grid(cl.cluster_grid(labels, grid), (W, H), nearest=True).astype(int)fig, axs = plt.subplots(1, K, figsize=(3 * K, 3))for c in range(K):    m = (lbl_full == c)[..., None]    axs[c].imshow((base * (0.2 + 0.8 * m)).astype('uint8'))    axs[c].set_title(f'cluster {c}'); axs[c].axis('off')plt.show()

## 5. Pick the chinee-apple cluster and localizeSet `TARGET_CLUSTER` to the id whose panel above best covers the weed. That oneinteger is the only human input in the whole pipeline.

In [ ]:
TARGET_CLUSTER = 3   # <-- change to the chinee-apple cluster id from 4bTHRESHOLD = 0.5result = cl.localize(feats, labels, centroids, TARGET_CLUSTER, grid, IMAGE,                     heat_threshold=THRESHOLD, min_area=64)from PIL import ImageDrawfig, axs = plt.subplots(1, 3, figsize=(18, 6))axs[0].imshow(result['heat'], cmap='inferno'); axs[0].set_title('P(chinee apple)'); axs[0].axis('off')axs[1].imshow(result['mask'], cmap='gray'); axs[1].set_title('mask'); axs[1].axis('off')boxed = result['base_image'].copy(); d = ImageDraw.Draw(boxed)for (x0, y0, x1, y1, _a) in result['boxes']:    d.rectangle([x0, y0, x1, y1], outline='lime', width=3)axs[2].imshow(boxed); axs[2].set_title(f"{len(result['boxes'])} boxes"); axs[2].axis('off')plt.show()cl.save_outputs(result, '/content/out', stem='tile')

## 6. From picks to a trained detectorThe KMeans map above is per-tile — its cluster ids don't carry over to newimages. Sections 7–8 promote the pick you just validated into a **trainedlinear probe**: pseudo-label patches from the cluster, fit a logistic-regressionhead on the frozen tokens, then score any tile densely. Same head as the`dinov3-colab` notebook, but with zero hand-drawn labels.

## 7. Stage 2 — train a probe from the cluster picks (no manual labels)The pick you validated in section 5 is the **only** supervision. Here we:1. Save the chosen cluster's centroid as a **reference** for chinee apple.2. Re-cluster each training tile and auto-assign the positive cluster as the   one whose centroid is closest (cosine) to that reference — this fixes the   fact that per-tile KMeans ids are arbitrary and not comparable across tiles.3. Pool `(patch features, binary label)` and fit the same logistic-regression   head used in the `dinov3-colab` notebook — labels come from clusters, not   hand-drawn crops.Upload 5–10 representative tiles. **Include the tile you just validated** so thereference cluster is well represented.

In [ ]:
import numpy as npfrom google.colab import files# Reference = the chinee-apple centroid you validated in section 5 (L2-normalized).REF_CENTROID = centroids[TARGET_CLUSTER] / (np.linalg.norm(centroids[TARGET_CLUSTER]) + 1e-8)print('Upload 5-10 training tiles (include the one you validated above)...')train_up = files.upload()TRAIN_TILES = list(train_up)X_parts, y_parts = [], []for path in TRAIN_TILES:    feats_i, grid_i = vf.patch_features(model, path, meta, img_size=768)    labels_i, centroids_i = vf.cluster_patches(feats_i, k=K)    # Per-tile cluster ids are arbitrary -> match by centroid to the reference.    cen_n = centroids_i / (np.linalg.norm(centroids_i, axis=1, keepdims=True) + 1e-8)    pos_cluster = int(np.argmax(cen_n @ REF_CENTROID))    y_i = (labels_i == pos_cluster).astype(int)    X_parts.append(feats_i); y_parts.append(y_i)    print(f'  {path}: pos cluster {pos_cluster} -> {int(y_i.sum())}/{len(y_i)} positive patches')X = np.concatenate(X_parts); y = np.concatenate(y_parts)print('dataset:', X.shape, '| positives:', int(y.sum()), '/', len(y))

In [ ]:
from sklearn.linear_model import LogisticRegressionfrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import classification_reportX_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)# Same frozen-feature linear probe as dinov3-colab, but y came from cluster picks.clf = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced')clf.fit(X_tr, y_tr)print(classification_report(y_te, clf.predict(X_te), target_names=['other', 'chinee apple']))

## 8. Dense detection — apply the trained probe to a new tileThe probe scores **every patch**, so it becomes a detector with no extratraining and no re-clustering. Unlike the raw KMeans map, this transfers totiles it never saw. Upload a fresh tile.

In [ ]:
from PIL import Image, ImageDrawtest_up = files.upload()TEST_IMAGE = next(iter(test_up))feats_t, grid_t = vf.patch_features(model, TEST_IMAGE, meta, img_size=768)pos_col = clf.classes_.tolist().index(1)            # column of the positive classprob_grid = clf.predict_proba(feats_t)[:, pos_col].reshape(grid_t)base = Image.open(TEST_IMAGE).convert('RGB'); W, H = base.sizeheat = vf.upsample_grid(prob_grid, (W, H), nearest=False)mask = (heat >= 0.5).astype('uint8')boxes = vf.boxes_from_mask(mask, min_area=64)boxed = base.copy(); d = ImageDraw.Draw(boxed)for (x0, y0, x1, y1, _a) in boxes:    d.rectangle([x0, y0, x1, y1], outline='lime', width=3)fig, axs = plt.subplots(1, 3, figsize=(18, 6))axs[0].imshow(base); axs[0].set_title('input'); axs[0].axis('off')axs[1].imshow(base); axs[1].imshow(heat, cmap='inferno', alpha=0.5)axs[1].set_title('P(chinee apple) - trained probe'); axs[1].axis('off')axs[2].imshow(boxed); axs[2].set_title(f'{len(boxes)} box(es)'); axs[2].axis('off')plt.show()

In [ ]:
import joblib# Persist the head + reference centroid (~KB). The backbone is never saved.PROBE_PATH = '/content/chinee_probe_dinov3vit.joblib'joblib.dump({'clf': clf, 'ref_centroid': REF_CENTROID, 'k': K,             'family': FAMILY, 'img_size': 768,             'classes': ['other', 'chinee apple']}, PROBE_PATH)print('saved ->', PROBE_PATH)# Reload later:  b = joblib.load(PROBE_PATH); clf = b['clf']; REF_CENTROID = b['ref_centroid']